In [1]:
import multiprocessing
multiprocessing.set_start_method("spawn", force=True)

import polars as pl
import numpy as np
import matplotlib.pyplot as plt

In [2]:
datapath = '../data/2010-2011 Solar home electricity data.csv'
# skip the first line in csv and read the next line as column
# then read the rest of the file and store as dataframe
df = pl.read_csv(datapath, skip_rows=1)
print(df)
print(df.columns)
episode_num = "episode_01"

shape: (269_735, 53)
┌──────────┬───────────┬──────────┬──────────────────────┬───┬───────┬───────┬───────┬───────┐
│ Customer ┆ Generator ┆ Postcode ┆ Consumption Category ┆ … ┆ 22:30 ┆ 23:00 ┆ 23:30 ┆ 0:00  │
│ ---      ┆ Capacity  ┆ ---      ┆ ---                  ┆   ┆ ---   ┆ ---   ┆ ---   ┆ ---   │
│ i64      ┆ ---       ┆ i64      ┆ str                  ┆   ┆ f64   ┆ f64   ┆ f64   ┆ f64   │
│          ┆ f64       ┆          ┆                      ┆   ┆       ┆       ┆       ┆       │
╞══════════╪═══════════╪══════════╪══════════════════════╪═══╪═══════╪═══════╪═══════╪═══════╡
│ 1        ┆ 3.78      ┆ 2076     ┆ GC                   ┆ … ┆ 0.378 ┆ 0.128 ┆ 0.078 ┆ 0.125 │
│ 1        ┆ 3.78      ┆ 2076     ┆ CL                   ┆ … ┆ 0.0   ┆ 0.0   ┆ 0.0   ┆ 1.075 │
│ 1        ┆ 3.78      ┆ 2076     ┆ GG                   ┆ … ┆ 0.0   ┆ 0.0   ┆ 0.0   ┆ 0.0   │
│ 1        ┆ 3.78      ┆ 2076     ┆ GC                   ┆ … ┆ 0.402 ┆ 0.142 ┆ 0.12  ┆ 0.111 │
│ 1        ┆ 3.78      ┆ 2076

In [2]:
datapath = '../data/2011-2012 Solar home electricity data v2.csv'
# skip the first line in csv and read the next line as column
# then read the rest of the file and store as dataframe
df = pl.read_csv(datapath, skip_rows=1)
print(df)
print(df.columns)
episode_num = "episode_02"

shape: (270_304, 54)
┌──────────┬───────────┬──────────┬──────────────────────┬───┬───────┬───────┬───────┬─────────────┐
│ Customer ┆ Generator ┆ Postcode ┆ Consumption Category ┆ … ┆ 23:00 ┆ 23:30 ┆ 0:00  ┆ Row Quality │
│ ---      ┆ Capacity  ┆ ---      ┆ ---                  ┆   ┆ ---   ┆ ---   ┆ ---   ┆ ---         │
│ i64      ┆ ---       ┆ i64      ┆ str                  ┆   ┆ f64   ┆ f64   ┆ f64   ┆ str         │
│          ┆ f64       ┆          ┆                      ┆   ┆       ┆       ┆       ┆             │
╞══════════╪═══════════╪══════════╪══════════════════════╪═══╪═══════╪═══════╪═══════╪═════════════╡
│ 1        ┆ 3.78      ┆ 2076     ┆ CL                   ┆ … ┆ 0.0   ┆ 0.0   ┆ 1.063 ┆ null        │
│ 1        ┆ 3.78      ┆ 2076     ┆ GC                   ┆ … ┆ 0.118 ┆ 0.219 ┆ 0.162 ┆ null        │
│ 1        ┆ 3.78      ┆ 2076     ┆ GG                   ┆ … ┆ 0.0   ┆ 0.0   ┆ 0.0   ┆ null        │
│ 1        ┆ 3.78      ┆ 2076     ┆ CL                   ┆ … ┆ 0.0   ┆

In [ ]:
datapath = '../data/2012-2013 Solar home electricity data v2.csv'
# skip the first line in csv and read the next line as column
# then read the rest of the file and store as dataframe
df = pl.read_csv(datapath, skip_rows=1)
print(df)
print(df.columns)
episode_num = "episode_03"

In [3]:
# we can get the training and testing customers from the csv file
training_customers = np.loadtxt('../data/training_customers.csv', dtype=int)
testing_customers = np.loadtxt('../data/testing_customers.csv', dtype=int)

In [ ]:
# alternatively, get all the unique customers as their own dataframes
customers = df['Customer'].unique()
# pick 80% of the random customers as training data
training_customers = np.random.choice(customers, int(0.8*len(customers)), replace=False)
# the rest of the customers are testing data
testing_customers = np.setdiff1d(customers, training_customers)

In [ ]:
# save the customers number to a csv file
np.savetxt('../data/training_customers.csv', training_customers, fmt='%s')
np.savetxt('../data/testing_customers.csv', testing_customers, fmt='%s')

In [4]:
from helper import transform_polars_df
# loop through each customer and use transform_polars_df to get the dataframe and store it in a list call dataset
training_dataset = []
for customer in training_customers:
    customer_df = df.filter(pl.col('Customer') == customer)
    try:
        newcustomerdf = transform_polars_df(customer_df, import_energy_price=0.23, export_energy_price=0.015, price_periods="7am – 10am | 4pm – 9pm", default_import_energy_price=0.15, default_export_energy_price=0.01)
    except Exception as e:
        print(f"Error with customer as training dataset: {customer}")
        print(e)
        break
    training_dataset.append(newcustomerdf)

testing_dataset = []
for customer in testing_customers:
    customer_df = df.filter(pl.col('Customer') == customer)
    try:
        newcustomerdf = transform_polars_df(customer_df, import_energy_price=0.23, export_energy_price=0.015, price_periods="7am – 10am | 4pm – 9pm", default_import_energy_price=0.15, default_export_energy_price=0.01)
    except Exception as e:
        print(f"Error with customer as testing dataset: {customer}")
        print(e)
        break
    testing_dataset.append(newcustomerdf)

In [ ]:
# provide std and mean on the different columns of the training dataset
testdf = testing_dataset[25]
# drop timestamp and time columns
testdf = testdf.drop(['Timestamp', 'Time'])
print(testdf.describe())

In [5]:
import gymnasium as gym

from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv
from stable_baselines3.common.env_checker import check_env
from EnergySimEnv import SolarBatteryEnv
from helper import make_env

testing_env_fns = [make_env(ds) for ds in testing_dataset]

num_step = None # pick the number of step for the simulation/none for full length
test_envs = [env_fn(num_step) for env_fn in testing_env_fns]

/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


In [6]:
import gymnasium as gym

from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv
from stable_baselines3.common.env_checker import check_env
from EnergySimEnv import SolarBatteryEnv
from helper import make_env
# Create a list of environment creation functions to build a vectorized environment.
training_env_fns = [make_env(ds) for ds in training_dataset]
#training_vec_env = DummyVecEnv(training_env_fns)
num_step = None # pick the number of step for the simulation/none for full length
train_envs = [env_fn(num_step) for env_fn in training_env_fns]

In [ ]:
# combine the test_envs and train_envs into a single list
combined_envs = test_envs + train_envs

In [7]:
selected_list = train_envs
if selected_list is test_envs:
    env_type = "test"
    env_fns = testing_env_fns
elif selected_list is train_envs:
    env_type = "train"
    env_fns = training_env_fns
else:
    env_type = "combined"
    env_fns = testing_env_fns + training_env_fns

In [8]:
from decision import Agent, run_episodes_parallel
rule_agent_kwargs = {
    'algorithm': 'rule'
}

# run episodes in the list in parallel using the rule-based agent on the training environments
episode_logs, incident_logs = run_episodes_parallel(Agent, selected_list, agent_kwargs=rule_agent_kwargs, max_workers=12, use_notebook_tqdm=False)

[INFO] Starting 240 episodes with max_workers=12


Episodes: 100%|██████████| 240/240 [01:26<00:00,  2.76it/s]


[START] Episode 9
Sim Complete
[DONE]  Episode 9 (Elapsed: 4.60 sec)
[START] Episode 20
Sim Complete
[DONE]  Episode 20 (Elapsed: 0.29 sec)
[START] Episode 23
Sim Complete
[DONE]  Episode 23 (Elapsed: 4.13 sec)
[START] Episode 34
Sim Complete
[DONE]  Episode 34 (Elapsed: 4.23 sec)
[START] Episode 44
Sim Complete
[DONE]  Episode 44 (Elapsed: 4.41 sec)
[START] Episode 57
Sim Complete
[DONE]  Episode 57 (Elapsed: 4.21 sec)
[START] Episode 68
Sim Complete
[DONE]  Episode 68 (Elapsed: 1.98 sec)
[START] Episode 73
Sim Complete
[DONE]  Episode 73 (Elapsed: 4.15 sec)
[START] Episode 84
Sim Complete
[DONE]  Episode 84 (Elapsed: 4.74 sec)
[START] Episode 97
Sim Complete
[DONE]  Episode 97 (Elapsed: 4.14 sec)
[START] Episode 109
Sim Complete
[DONE]  Episode 109 (Elapsed: 4.33 sec)
[START] Episode 122
Sim Complete
[DONE]  Episode 122 (Elapsed: 4.14 sec)
[START] Episode 135
Sim Complete
[DONE]  Episode 135 (Elapsed: 4.28 sec)
[START] Episode 147
Sim Complete
[DONE]  Episode 147 (Elapsed: 0.18 sec)


In [9]:
dfs_with_id = [df.with_columns(pl.lit(i).alias("episode_id")) for i, df in enumerate(episode_logs)]
rule_all_logs = pl.concat(dfs_with_id)
file_name = f"../data/rule_{env_type}_{episode_num}_logs.parquet"
rule_all_logs.write_parquet(file_name)



In [10]:
# Incident logs
incident_dfs_with_id = [
    df.with_columns(pl.lit(i).alias("episode_id"))
    for i, df in enumerate(incident_logs)
    if df.height > 0
]
try:
    rule_incident_logs = pl.concat(incident_dfs_with_id)
    incident_file_name = f"../data/rule_{env_type}_{episode_num}_incident_logs.parquet"
    rule_incident_logs.write_parquet(incident_file_name)
except ValueError:
    print("[INFO] No incident logs to save.")

[INFO] No incident logs to save.


In [11]:
# check for schema mismatches if there are any
# Collect all schemas
schemas = [df.schema for df in dfs_with_id]

# Find the most common schema (assume it's the correct one)
from collections import Counter
schema_counts = Counter([tuple(sorted(s.items())) for s in schemas])
most_common_schema = dict(schema_counts.most_common(1)[0][0])

# Print out indices and details of DataFrames with mismatched schemas
for i, schema in enumerate(schemas):
    if dict(sorted(schema.items())) != most_common_schema:
        print(f"DF {i} schema mismatch:")
        print("Schema:", schema)
        print("Difference:", set(schema.items()) ^ set(most_common_schema.items()))

In [12]:
from decision import Agent, run_episodes_parallel, run_single
# Initialize environments and SDP agent parameters
sdp_agent_kwargs = {
    "algorithm": "sdp",
    "horizon": 48,                  # planning horizon (steps)
    "soc_resolution": 20,           # SoC discretization
    "action_resolution": 41,        # discrete actions (best ≈ 2*soc_resolution + 1)
    "use_monte_carlo": True,
    "mc_samples": 200,
    "mc_seed": None,
}

# Run a single episode for timing test
#sdp_single_log = run_single(Agent, combined_envs[0], agent_kwargs=sdp_agent_kwargs, render=False, display_progress=True)


# Run all episodes in parallel
sdp_episode_logs, sdp_incident_logs = run_episodes_parallel(Agent, selected_list, agent_kwargs=sdp_agent_kwargs, max_workers=12, use_notebook_tqdm=False)

[INFO] Starting 240 episodes with max_workers=12


Episodes: 100%|██████████| 240/240 [10:38:22<00:00, 159.59s/it]  


[START] Episode 1
Sim Complete
[DONE]  Episode 1 (Elapsed: 2063.42 sec)
[START] Episode 24
Sim Complete
[DONE]  Episode 24 (Elapsed: 2054.57 sec)
[START] Episode 37
Sim Complete
[DONE]  Episode 37 (Elapsed: 2058.96 sec)
[START] Episode 49
Sim Complete
[DONE]  Episode 49 (Elapsed: 2070.11 sec)
[START] Episode 62
Sim Complete
[DONE]  Episode 62 (Elapsed: 2039.90 sec)
[START] Episode 74
Sim Complete
[DONE]  Episode 74 (Elapsed: 2048.60 sec)
[START] Episode 87
Sim Complete
[DONE]  Episode 87 (Elapsed: 2028.64 sec)
[START] Episode 99
Sim Complete
[DONE]  Episode 99 (Elapsed: 2050.91 sec)
[START] Episode 111
Sim Complete
[DONE]  Episode 111 (Elapsed: 2025.55 sec)
[START] Episode 124
Sim Complete
[DONE]  Episode 124 (Elapsed: 2037.95 sec)
[START] Episode 137
Sim Complete
[DONE]  Episode 137 (Elapsed: 2027.53 sec)
[START] Episode 150
Sim Complete
[DONE]  Episode 150 (Elapsed: 2038.09 sec)
[START] Episode 162
Sim Complete
[DONE]  Episode 162 (Elapsed: 2013.54 sec)
[START] Episode 174
Sim Comple

In [13]:
dfs_with_id = [df.with_columns(pl.lit(i).alias("episode_id")) for i, df in enumerate(sdp_episode_logs)]
sdp_all_logs = pl.concat(dfs_with_id)
file_name = f"../data/sdp_{env_type}_{episode_num}_logs.parquet"
sdp_all_logs.write_parquet(file_name)

In [14]:
# incident logs
incident_dfs_with_id = [
    df.with_columns(pl.lit(i).alias("episode_id"))
    for i, df in enumerate(sdp_incident_logs)
    if df.height > 0
]
try:
    sdp_incident_logs = pl.concat(incident_dfs_with_id)
    incident_file_name = f"../data/sdp_{env_type}_{episode_num}_incident_logs.parquet"
    sdp_incident_logs.write_parquet(incident_file_name)
except ValueError:
    print("[INFO] No incident logs to save.")


[INFO] No incident logs to save.


In [15]:
from decision import Agent, run_episodes_parallel, run_single
mrdp_agent_kwargs = {
    'algorithm': 'mrdp',
    'soc_resolution': 20,           # fallback/default for single-horizon
    'action_resolution': 41,        # fallback/default for single-horizon
    'subhorizon_specs': [
        {
            'start': 0,
            'length': 12,           # e.g. 6 hours at 30-min steps
            'soc_resolution': 20,   # fine SoC discretization
            'action_resolution': 41,# fine action discretization
            'step_duration': 0.5    # hours per step (30 min)
        },
        {
            'start': 12,
            'length': 72,           # e.g. 36 hours at 30-min steps
            'soc_resolution': 8,    # coarse SoC discretization
            'action_resolution': 17, # coarse action discretization
            'step_duration': 0.5    # hours per step (30 min)
        }
    ],
    'use_monte_carlo': True,
    'mc_samples': 200,
    'mc_seed': None,
}

#sdp_single_log = run_single(Agent, combined_envs[0], agent_kwargs=mrdp_agent_kwargs, render=False, display_progress=True)

# Run all episodes in parallel using MRDP

mrdp_episode_logs, mrdp_incident_logs = run_episodes_parallel(
    Agent, selected_list, agent_kwargs=mrdp_agent_kwargs, max_workers=12, use_notebook_tqdm=False
)

[INFO] Starting 240 episodes with max_workers=12


Episodes: 100%|██████████| 240/240 [10:11:11<00:00, 152.80s/it]  


[START] Episode 4
Sim Complete
[DONE]  Episode 4 (Elapsed: 1916.90 sec)
[START] Episode 21
Sim Complete
[DONE]  Episode 21 (Elapsed: 1916.82 sec)
[START] Episode 33
Sim Complete
[DONE]  Episode 33 (Elapsed: 1934.64 sec)
[START] Episode 46
Sim Complete
[DONE]  Episode 46 (Elapsed: 1927.25 sec)
[START] Episode 59
Sim Complete
[DONE]  Episode 59 (Elapsed: 1929.52 sec)
[START] Episode 73
Sim Complete
[DONE]  Episode 73 (Elapsed: 1920.89 sec)
[START] Episode 85
Sim Complete
[DONE]  Episode 85 (Elapsed: 1906.31 sec)
[START] Episode 98
Sim Complete
[DONE]  Episode 98 (Elapsed: 1906.69 sec)
[START] Episode 110
Sim Complete
[DONE]  Episode 110 (Elapsed: 1910.43 sec)
[START] Episode 123
Sim Complete
[DONE]  Episode 123 (Elapsed: 1910.16 sec)
[START] Episode 136
Sim Complete
[DONE]  Episode 136 (Elapsed: 1911.79 sec)
[START] Episode 148
Sim Complete
[DONE]  Episode 148 (Elapsed: 1910.90 sec)
[START] Episode 160
Sim Complete
[DONE]  Episode 160 (Elapsed: 1910.42 sec)
[START] Episode 172
Sim Comple

In [16]:
dfs_with_id = [df.with_columns(pl.lit(i).alias("episode_id")) for i, df in enumerate(mrdp_episode_logs)]
mrdp_episode_logs = pl.concat(dfs_with_id)
file_name = f"../data/mrdp_{env_type}_{episode_num}_logs.parquet"
mrdp_episode_logs.write_parquet(file_name)

In [17]:
incident_dfs_with_id = [
    df.with_columns(pl.lit(i).alias("episode_id"))
    for i, df in enumerate(mrdp_incident_logs)
    if df.height > 0
]
try:
    mrdp_incident_logs = pl.concat(incident_dfs_with_id)
    incident_file_name = f"../data/mrdp_{env_type}_{episode_num}_incident_logs.parquet"
    mrdp_incident_logs.write_parquet(incident_file_name)
except ValueError:
    print("[INFO] No incident logs to save.")


[INFO] No incident logs to save.


In [18]:
from decision import Agent, run_episodes_parallel

oracle_agent_kwargs = {
    "algorithm": "oracle",
    "horizon": 48,              # 48 steps (~24 h @ 0.5h/step)
    "soc_resolution": 20,       # SoC discretization levels
    "action_resolution": 41    # ≈ 2*soc_resolution + 1 (fine enough)
}

# Run all episodes in parallel using the oracle agent
oracle_episode_logs, oracle_incident_logs = run_episodes_parallel(
    Agent,
    selected_list[64:89],  # your list of environments
    agent_kwargs=oracle_agent_kwargs,
    max_workers=14,
    use_notebook_tqdm=False
)

[INFO] Starting 25 episodes with max_workers=14


Episodes: 100%|██████████| 25/25 [5:58:20<00:00, 860.01s/it]    


[START] Episode 4
Sim Complete
[DONE]  Episode 4 (Elapsed: 11638.47 sec)
[START] Episode 9
Sim Complete
[DONE]  Episode 9 (Elapsed: 12294.55 sec)
[START] Episode 2
Sim Complete
[DONE]  Episode 2 (Elapsed: 13152.63 sec)
[START] Episode 0
Sim Complete
[DONE]  Episode 0 (Elapsed: 10229.74 sec)
[START] Episode 14
Sim Complete
[DONE]  Episode 14 (Elapsed: 9755.94 sec)
[START] Episode 6
Sim Complete
[DONE]  Episode 6 (Elapsed: 10833.51 sec)
[START] Episode 16
Sim Complete
[DONE]  Episode 16 (Elapsed: 9527.46 sec)
[START] Episode 12
Sim Complete
[DONE]  Episode 12 (Elapsed: 10935.59 sec)
[START] Episode 17
Sim Complete
[DONE]  Episode 17 (Elapsed: 9458.93 sec)
[START] Episode 5
Sim Complete
[DONE]  Episode 5 (Elapsed: 11165.07 sec)
[START] Episode 19
Sim Complete
[DONE]  Episode 19 (Elapsed: 9365.69 sec)
[START] Episode 3
Sim Complete
[DONE]  Episode 3 (Elapsed: 10684.88 sec)
[START] Episode 15
Sim Complete
[DONE]  Episode 15 (Elapsed: 9878.79 sec)
[START] Episode 10
Sim Complete
[DONE]  Epis

In [19]:
dfs_with_id = [df.with_columns(pl.lit(i).alias("episode_id")) for i, df in enumerate(oracle_episode_logs)]
oracle_episode_logs = pl.concat(dfs_with_id)
file_name = f"../data/oracle_{env_type}_{episode_num}_64-89_logs.parquet"
oracle_episode_logs.write_parquet(file_name)

In [20]:
incident_dfs_with_id = [
    df.with_columns(pl.lit(i).alias("episode_id"))
    for i, df in enumerate(oracle_incident_logs)
    if df.height > 0
]
try:
    oracle_incident_logs = pl.concat(incident_dfs_with_id)
    incident_file_name = f"../data/oracle_{env_type}_{episode_num}_incident_logs.parquet"
    oracle_incident_logs.write_parquet(incident_file_name)
except ValueError:
    print("[INFO] No incident logs to save.")

[INFO] No incident logs to save.


In [21]:
from decision import Agent, run_sb3_model_on_vec_env
from stable_baselines3 import PPO, A2C, DDPG, SAC, TD3
from helper import flatten_episode_data

# (only needed if you ever switch to SubprocVecEnv on Linux/notebooks)
multiprocessing.set_start_method("forkserver", force=True)

# Utility to yield batches from a list
def batchify(lst, batch_size):
    """Yield successive batches from lst of size batch_size."""
    for i in range(0, len(lst), batch_size):
        yield lst[i:i + batch_size]

from stable_baselines3.common.vec_env import SubprocVecEnv

batch_size = 64  # Set your desired batch size

In [22]:
sac_model = SAC.load("../models/sac_model.zip")
# Run test episodes in parallel in batches
all_episode_logs = []
for batch_num, env_fns_batch in enumerate(batchify(env_fns, batch_size)):
    print(f"Processing batch {batch_num+1}")
    vec_env = SubprocVecEnv(env_fns_batch)
    # Run your model on this batch
    episode_logs = run_sb3_model_on_vec_env(sac_model, vec_env)
    all_episode_logs.extend(episode_logs)
    vec_env.close()

sac_logs = flatten_episode_data(all_episode_logs)
file_name = f"../data/sac_{env_type}_{episode_num}_logs.parquet"
sac_logs.write_parquet(file_name)

Processing batch 1


/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWar

Processing batch 2


/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWar

Processing batch 3


/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWar

Processing batch 4


/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWar

In [23]:
PPO_model = PPO.load("../models/ppo_model.zip")
# Run test episodes in parallel in batches
all_episode_logs = []
for batch_num, env_fns_batch in enumerate(batchify(env_fns, batch_size)):
    print(f"Processing batch {batch_num+1}")
    vec_env = SubprocVecEnv(env_fns_batch)
    # Run your model on this batch
    episode_logs = run_sb3_model_on_vec_env(PPO_model, vec_env)
    all_episode_logs.extend(episode_logs)
    vec_env.close()

ppo_logs = flatten_episode_data(all_episode_logs)
file_name = f"../data/ppo_{env_type}_{episode_num}_logs.parquet"
ppo_logs.write_parquet(file_name)

/usr/local/lib/python3.10/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


Processing batch 1


/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWar

Processing batch 2


/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWar

Processing batch 3


/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWar

Processing batch 4


/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWar

In [24]:
a2c_model = A2C.load("../models/a2c_model.zip")
# Run test episodes in parallel in batches
all_episode_logs = []
for batch_num, env_fns_batch in enumerate(batchify(env_fns, batch_size)):
    print(f"Processing batch {batch_num+1}")
    vec_env = SubprocVecEnv(env_fns_batch)
    # Run your model on this batch
    episode_logs = run_sb3_model_on_vec_env(a2c_model, vec_env)
    all_episode_logs.extend(episode_logs)
    vec_env.close()

a2c_logs = flatten_episode_data(all_episode_logs)
file_name = f"../data/a2c_{env_type}_{episode_num}_logs.parquet"
a2c_logs.write_parquet(file_name)

/usr/local/lib/python3.10/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run A2C on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


Processing batch 1


/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWar

Processing batch 2


/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWar

Processing batch 3


/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWar

Processing batch 4


/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWar

In [25]:
ddpg_model = DDPG.load("../models/ddpg_model.zip")
# Run test episodes in parallel in batches
all_episode_logs = []
for batch_num, env_fns_batch in enumerate(batchify(env_fns, batch_size)):
    print(f"Processing batch {batch_num+1}")
    vec_env = SubprocVecEnv(env_fns_batch)
    # Run your model on this batch
    episode_logs = run_sb3_model_on_vec_env(ddpg_model, vec_env)
    all_episode_logs.extend(episode_logs)
    vec_env.close()
ddpg_logs = flatten_episode_data(all_episode_logs)
file_name = f"../data/ddpg_{env_type}_{episode_num}_logs.parquet"
ddpg_logs.write_parquet(file_name)

Processing batch 1


/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWar

Processing batch 2


/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWar

Processing batch 3


/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWar

Processing batch 4


/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWar

In [26]:
td3_model = TD3.load("../models/td3_model.zip")
# Run test episodes in parallel in batches
all_episode_logs = []
for batch_num, env_fns_batch in enumerate(batchify(env_fns, batch_size)):
    print(f"Processing batch {batch_num+1}")
    vec_env = SubprocVecEnv(env_fns_batch)
    # Run your model on this batch
    episode_logs = run_sb3_model_on_vec_env(td3_model, vec_env)
    all_episode_logs.extend(episode_logs)
    vec_env.close()
td3_logs = flatten_episode_data(all_episode_logs)
file_name = f"../data/td3_{env_type}_{episode_num}_logs.parquet"
td3_logs.write_parquet(file_name)

Processing batch 1


/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWar

Processing batch 2


/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWar

Processing batch 3


/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWar

Processing batch 4


/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWar

In [ ]:
# to get the best RTG value, analysis of the distribution of total episode rewards in the dataset is needed


In [ ]:
from decision import Agent, run_episodes_parallel, run_single
from decision_transformer import DecisionTransformer
import json
import os

import torch
# check if GPU is available
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

with open('../models/decision_transformer_model_kwargs.json', 'r') as f:
    model_kwargs = json.load(f)

model = DecisionTransformer(**model_kwargs)

# Prefer loading from the training checkpoint if present (it contains return_scale).
# Otherwise load weights and (if available) restore return_scale from the sidecar `.meta.json`.
weights_path = '../models/dt_model.pt'
checkpoint_path = '../models/dt_model_checkpoint.pt'
load_path = checkpoint_path if os.path.exists(checkpoint_path) else weights_path

model.load_from_checkpoint(load_path, map_location=device)
print(f"Loaded DT from {load_path}; model.return_scale={getattr(model, 'return_scale', None)}")

model = model.to(device)
model.eval()

rtg = 4508964.69

dt_agent_kwargs = {
    'algorithm': 'dt',
    'model': model,
    'rtg_value': rtg
}

# check for nan in model parameters
bad_params = [name for name, p in model.named_parameters() if torch.isnan(p).any() or torch.isinf(p).any()]
print("bad params:", bad_params)

In [ ]:
episode_log = run_episodes_parallel(Agent, test_envs, agent_kwargs=dt_agent_kwargs, max_workers=2, use_notebook_tqdm=False)

dfs_with_id = [df.with_columns(pl.lit(i).alias("episode_id")) for i, df in enumerate(episode_log)]
dt_logs = pl.concat(dfs_with_id)
file_name = f"../data/dt_rtg{int(rtg)}_{env_type}_{episode_num}_logs.parquet"
dt_logs.write_parquet(file_name)